In [ ]:
import pennylane as qml
from pennylane import numpy as np

## Defining a quantum circuit and setting a device

A quantum cicrcuit (quantum function), is a Python function that contains a list of gates that may depend on some *parameters* and act on some *wires*, i.e. qubits.
In Pennylane, all wires (qubits) are initialized to |0>

Example of a generic quantum circuit/function

~~~
def quantum_function(parameters)
    qml.gate1(parameters,wires)
    qml.gate2(wires)
~~~    

In [3]:
def quantum_function(theta):
    qml.RX(theta, wires=0) #applies X to qubit 0
    qml.PauliY(wires=1) #applies Y to qubit 1
    qml.Hadamard(wires=0) #applies H to qubit 0
    qml.Hadamard(wires=1) #applies H to qubit 1

    return qml.state() #returns a complex np.array, representing the quantum state

To obtain a result from *quantum_function(theta)*, Pennylane requires the specification of a device in which the quantum circuit will run

The tutorial talks about three devices:
- *default.qubit* : vanilla qubit quantum device for noiseless circuits
- *lightning.qubit* : a fast noiseless qubit device
- *default.mixed* : a qubit device that allows noisy gates (states represented as density operators)

In [4]:
dev = qml.device("default.qubit", wires=2)

## Creating a **QNode** 

To obtain a result, we also need to pair a quantum circuit with a device.

Such connection is called **QNode**

There are two ways of doing a **QNode**:
1. using the `@qml.qnode` decorator (placed on top of the definition of the quantum function)
2. using the `qml.QNode` function

#### Creating a **QNode** with `@qml.qnode`

In [21]:
dev1 = qml.device("default.qubit", wires=1)

@qml.qnode(dev1)
def app_X(theta):
    qml.RX(theta,wires=0)

    return qml.state()

print(app_X(np.pi/4))

[0.92387953+0.j         0.        -0.38268343j]


#### Creating a **QNode** with `qml.QNode`

In [22]:
dev2 = qml.device("default.qubit", wires=1)

def app_X1(theta):
    qml.RX(theta, wires=0)

    return qml.state()

qnode = qml.QNode(app_X1,dev2)

print(qnode(np.pi/4))

[0.92387953+0.j         0.        -0.38268343j]


## Custom wires

With Pennylane we can customize wires, i.e. we have different ways to fill the `wires` field
1. Default

    `dev = qml.device("default.qubit", wires=3)`
    
    Circuit with three qubits, such that |abc> --> |012>

2. Using an array

    `dev = qml.device("default.qubit", wires=[0,1,2])`

    Circuit with three qubits, such that |abc> --> |012>

3. Using range

    `dev = qml.device("default.qubit", wires = range(3))`

    Circuit with three qubits, such that |abc> --> |012>

4. Using an array to change the order of the wires

    `dev = qml.device("default.qubit", wires=[2,0,1])`

    Circuit with three qubits, such that |abc> --> |201>

5. Use strings to name wires

    `dev = qml.device("default.qubit", wires = ["control", "target"])`

    Circuit with two qubits, such that the first qubit is called "control" and the second qubit is called "target"

In [27]:
dev_ct = qml.device("default.qubit", wires = ["control", "target"])

@qml.qnode(dev_ct)
def max_ent():
    qml.Hadamard(wires = "control")
    qml.CNOT(wires = ["control", "target"])

    return qml.state()

print(max_ent())

[0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]


The following code is not correct in Pennylane because wire labels are fixed at QNode construction time, not passed dynamically as function arguments.

In other words, wire labels are static and should be known when defining QNode

~~~
dev_ct = qml.device("default.qubit", wires = ["control", "target"])
@qml.qnode(dev_ct)
def max_ent1(c,t):
    qml.Hadamard(c)
    qml.CNOT(c,t)

    return qml.state()

print(max_ent1("control", "target"))
~~~

## Quantum functions as subcircuits

In [37]:
def subcircuit1(angle):
    qml.RX(angle, wires=0)
    qml.PauliY(wires=1)

def subcircuit2():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0,1])

def full_circuit(theta,phi):
    subcircuit1(theta)
    subcircuit2()
    subcircuit1(phi)

    return qml.state() #this was added to create a QNode in the next code block

theta=0.3
phi=0.2
print(qml.draw(full_circuit)(theta,phi))

0: ──RX(0.30)──H─╭●──RX(0.20)─┤  State
1: ──Y───────────╰X──Y────────┤  State


In [38]:
dev_full_circuit = qml.device("default.qubit",wires=2)

qnode_full_circuit = qml.QNode(full_circuit,dev_full_circuit)

print(qnode_full_circuit(0.3,0.2))

[ 0.69567381-0.10514081j -0.01054927+0.0698002j  -0.01054927-0.0698002j
 -0.69567381-0.10514081j]


#### Using a list of wires `wire_list` to change the order in which qubits are applied in the same circuit

In [42]:
def subcircuit_1(angle, wire_list):
    qml.RX(angle, wires=wire_list[0])
    qml.PauliY(wires=wire_list[1])

def subcircuit_2(wire_list):
    qml.Hadamard(wires=wire_list[0])
    qml.CNOT(wires=[wire_list[0], wire_list[1]])

@qml.qnode(dev)
def full_circ(theta, phi):
    subcircuit_1(theta,[0,1])
    subcircuit_2([0,1])
    subcircuit_1(phi,[1,0])

    return qml.state()

print(qml.draw(full_circ)(0.3,0.2))

0: ──RX(0.30)──H─╭●──Y────────┤  State
1: ──Y───────────╰X──RX(0.20)─┤  State
